# PDF to Chroma Pipeline

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\gandh\AppData\Local\Temp\ipykernel_18412\2843940200.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Prepare Paths and Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebook":
    project_root = project_root.parent

project_root

WindowsPath('e:/Machine Learning and Data Science/Advanced-RAGs-Detailed/04 Vector Stores')

In [4]:
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

PDF path: e:\Machine Learning and Data Science\Advanced-RAGs-Detailed\04 Vector Stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
Collection name: rag-pipeline
Persist directory: e:\Machine Learning and Data Science\Advanced-RAGs-Detailed\04 Vector Stores\chroma_langchain_db


In [5]:
load_dotenv()

True

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

### Helper

In [7]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Load the PDF

In [8]:
loader = PyPDFLoader(str(pdf_path))

In [9]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 6


In [10]:
docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'e:\\Machine Learning and Data Science\\Advanced-RAGs-Detailed\\04 Vector Stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo\nFor a long time, the most common experience with AI felt theatrical. You typed a question, the model\nanswered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a

## 4. Split the docs into chunks

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [14]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


In [15]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

First chunk preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'e:\\Machine Learning and Data Science\\Advanced-RAGs-Detailed\\04 Vector Stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 5. Store chunks in Chroma

In [16]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory)
)


print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


## 6. Retrieve Relevant chunks

In [17]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [20]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=e:\Machine Learning and Data Science\Advanced-RAGs-Detailed\04 Vector Stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=2 | source=e:\Machine Learning and Data Science\Advanced-RAGs-Detailed\04 Vector Stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=Page 3
Memory is not just a feature, it is a design decision
When people first hear that agents need memory, they sometimes imagine a giant transcript that the model
carries forever. In practice, memory is much more selective than that. Good systems decide what should stay
3. page=0 | source=e:\Machine Learni

In [23]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print()


Score: 0.4222
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform

Score: 0.4845
Content preview: Page 3
Memory is not just a feature, it is a design decision
When people first hear that agents need memory, they sometimes imagine a giant transcript that the model
carries forever. In practice, memory is much more selective than that. Good systems decide what should stay

